In [14]:
import pandas as pd
import os

# ==========================================================
# 1. LOAD DATA
# ==========================================================
eda_df = pd.read_csv("Data_Corpus/final/eda_dataset_final.csv")

eda_df["datetime"] = pd.to_datetime(eda_df["datetime"])
eda_df = eda_df.sort_values("datetime").reset_index(drop=True)
eda_df.head()

# ==========================================================
# 2. Create a Solar Modelling Dataset
# ==========================================================
solar_df = eda_df.copy()

# ==========================================================
# 3.Create Solar Lag Features
# ==========================================================
solar_df["solar_lag_1"] = solar_df["generation_photovoltaics_mwh"].shift(1)

solar_df["solar_lag_6"] = solar_df["generation_photovoltaics_mwh"].shift(6)

solar_df["solar_lag_24"] = solar_df["generation_photovoltaics_mwh"].shift(24)

solar_df["solar_lag_72"] = solar_df["generation_photovoltaics_mwh"].shift(72)

solar_df["solar_lag_168"] = solar_df["generation_photovoltaics_mwh"].shift(168)


# ==========================================================
# 4.Create Rolling Features
# ==========================================================
solar_df["solar_roll_mean_24"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

solar_df["solar_roll_mean_168"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

solar_df["solar_roll_std_24"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(1)
    .rolling(window=24)
    .std()
)



# ==========================================================
# 5.Create Forecast Targets
# ==========================================================
solar_df["target_h1"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(-1)
)

solar_df["target_h6"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(-6)
)

solar_df["target_h24"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(-24)
)

solar_df["target_h72"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(-72)
)

solar_df["target_h168"] = (
    solar_df["generation_photovoltaics_mwh"]
    .shift(-168)
)

# ==========================================================
# 6.Check Missing Values & Remove Rows That Cannot Be Used
# ==========================================================
print(" Check missing values: ")
print(solar_df.shape)
print(solar_df.isna().sum().sort_values(ascending=False).head(20))


solar_df = solar_df.dropna().reset_index(drop=True)

print(" Again Check missing values: ")
print(solar_df.shape)
print(solar_df.isna().sum().sort_values(ascending=False).head(20))

# ==========================================================
# 7.Verify Dataset
# ==========================================================

print(solar_df.shape)

print(solar_df[    [        "solar_lag_1",        "solar_lag_24",        "solar_roll_mean_24",        "target_h24"    ]].head())


 Check missing values: 
(98580, 54)
target_h168                       168
solar_roll_mean_168               168
solar_lag_168                     168
solar_lag_72                       72
target_h72                         72
target_h24                         24
solar_lag_24                       24
solar_roll_mean_24                 24
solar_roll_std_24                  24
solar_lag_6                         6
target_h6                           6
solar_lag_1                         1
target_h1                           1
generation_lignite_mwh              0
generation_other_renewable_mwh      0
generation_nuclear_mwh              0
datetime                            0
generation_biomass_mwh              0
generation_hydropower_mwh           0
generation_wind_offshore_mwh        0
dtype: int64
 Again Check missing values: 
(98244, 54)
datetime                               0
generation_biomass_mwh                 0
generation_hydropower_mwh              0
generation_wind_offshore_m

In [15]:
lag_cols = [
    "solar_lag_1",
    "solar_lag_6",
    "solar_lag_24",
    "solar_lag_72",
    "solar_lag_168"
]

solar_df[lag_cols + ["target_h24"]].corr()["target_h24"].sort_values(ascending=False)

target_h24       1.000000
solar_lag_24     0.925520
solar_lag_1      0.908394
solar_lag_72     0.905682
solar_lag_168    0.891448
solar_lag_6      0.073129
Name: target_h24, dtype: float64

# Wind

In [16]:
wind_on_df = eda_df.copy()
######################################################
wind_on_df["wind_on_lag_1"] = (
    wind_on_df["generation_wind_onshore_mwh"].shift(1)
)

wind_on_df["wind_on_lag_6"] = (
    wind_on_df["generation_wind_onshore_mwh"].shift(6)
)

wind_on_df["wind_on_lag_24"] = (
    wind_on_df["generation_wind_onshore_mwh"].shift(24)
)

wind_on_df["wind_on_lag_72"] = (
    wind_on_df["generation_wind_onshore_mwh"].shift(72)
)

wind_on_df["wind_on_lag_168"] = (
    wind_on_df["generation_wind_onshore_mwh"].shift(168)
)

###################################################
wind_on_df["target_h24"] = (
    wind_on_df["generation_wind_onshore_mwh"]
    .shift(-24)
)

####################################
wind_on_df = wind_on_df.dropna().reset_index(drop=True)

##########################################
lag_cols = [
    "wind_on_lag_1",
    "wind_on_lag_6",
    "wind_on_lag_24",
    "wind_on_lag_72",
    "wind_on_lag_168"
]

wind_on_corr = (
    wind_on_df[lag_cols + ["target_h24"]]
    .corr()["target_h24"]
    .sort_values(ascending=False)
)

print(wind_on_corr)



target_h24         1.000000
wind_on_lag_1      0.563525
wind_on_lag_6      0.493486
wind_on_lag_24     0.407025
wind_on_lag_72     0.264540
wind_on_lag_168    0.210120
Name: target_h24, dtype: float64


In [17]:
wind_off_df = eda_df.copy()

wind_off_df["wind_off_lag_1"] = (
    wind_off_df["generation_wind_offshore_mwh"].shift(1)
)

wind_off_df["wind_off_lag_6"] = (
    wind_off_df["generation_wind_offshore_mwh"].shift(6)
)

wind_off_df["wind_off_lag_24"] = (
    wind_off_df["generation_wind_offshore_mwh"].shift(24)
)

wind_off_df["wind_off_lag_72"] = (
    wind_off_df["generation_wind_offshore_mwh"].shift(72)
)

wind_off_df["wind_off_lag_168"] = (
    wind_off_df["generation_wind_offshore_mwh"].shift(168)
)

wind_off_df["target_h24"] = (
    wind_off_df["generation_wind_offshore_mwh"]
    .shift(-24)
)

wind_off_df = wind_off_df.dropna().reset_index(drop=True)

lag_cols = [
    "wind_off_lag_1",
    "wind_off_lag_6",
    "wind_off_lag_24",
    "wind_off_lag_72",
    "wind_off_lag_168"
]

wind_off_corr = (
    wind_off_df[lag_cols + ["target_h24"]]
    .corr()["target_h24"]
    .sort_values(ascending=False)
)

print(wind_off_corr)

target_h24          1.000000
wind_off_lag_1      0.490589
wind_off_lag_6      0.440172
wind_off_lag_24     0.375015
wind_off_lag_72     0.298743
wind_off_lag_168    0.288705
Name: target_h24, dtype: float64


In [20]:
def create_forecasting_dataset(
    df,
    target_col,
    horizons=[1, 6, 24, 72, 168],
    lags=[1, 6, 24, 72, 168]
):

    data = df.copy()

    # -----------------------
    # Lag Features
    # -----------------------
    for lag in lags:
        data[f"{target_col}_lag_{lag}"] = (
            data[target_col].shift(lag)
        )

    # -----------------------
    # Rolling Features
    # -----------------------
    data[f"{target_col}_roll_mean_24"] = (
        data[target_col]
        .shift(1)
        .rolling(24)
        .mean()
    )

    data[f"{target_col}_roll_mean_168"] = (
        data[target_col]
        .shift(1)
        .rolling(168)
        .mean()
    )

    data[f"{target_col}_roll_std_24"] = (
        data[target_col]
        .shift(1)
        .rolling(24)
        .std()
    )

    # -----------------------
    # Future Targets
    # -----------------------
    for h in horizons:
        data[f"target_h{h}"] = (
            data[target_col]
            .shift(-h)
        )

    data = data.dropna().reset_index(drop=True)

    return data

############################################
solar_model_df = create_forecasting_dataset(    eda_df,    target_col="generation_photovoltaics_mwh")
wind_on_model_df = create_forecasting_dataset(    eda_df,    target_col="generation_wind_onshore_mwh")
wind_off_model_df = create_forecasting_dataset(    eda_df,    target_col="generation_wind_offshore_mwh")

# Create folder if it doesn't exist
import os

os.makedirs("Data_Corpus/final/modelling", exist_ok=True)

# Save datasets
solar_model_df.to_csv(
    "Data_Corpus/final/modelling/solar_model_dataset.csv",
    index=False
)

wind_on_model_df.to_csv(
    "Data_Corpus/final/modelling/wind_onshore_model_dataset.csv",
    index=False
)

wind_off_model_df.to_csv(
    "Data_Corpus/final/modelling/wind_offshore_model_dataset.csv",
    index=False
)

print("Datasets saved successfully.")


Datasets saved successfully.
